In [1]:
import geopandas as gpd

json_df = gpd.read_file("./US_Petroleum_Pipelines.geojson")
json_df

,OBJECTID,Opername,Pipename,Source,Type,Notes,ARTIFICIAL,MASTER_OID,commodity,Volume,Capacity,VCR,Shape_Length,Mode_Type,Length,geometry
0,1,NUSTAR ENERGY,Valley Pipeline,EIA,Petroleum Product,None,0,299.0,Petroleum Products,None,None,None,1496.712523,pipeline_prod,0.930012,"MULTILINESTRING ((-97.38949 25.96387, -97.3921..."
1,2,TRANSMONTAIGNE PARTNERS,Diamondback Pipeline,EIA,Petroleum Product,None,0,300.0,Petroleum Products,None,None,None,1934.748751,pipeline_prod,1.202195,"MULTILINESTRING ((-97.38299 25.96636, -97.3839..."
2,3,NUSTAR ENERGY,Valley Pipeline,EIA,Petroleum Product,None,0,301.0,Petroleum Products,None,None,None,55.870089,pipeline_prod,0.034716,"MULTILINESTRING ((-97.39214 25.97691, -97.3922..."
3,4,TRANSMONTAIGNE PARTNERS,Diamondback Pipeline,EIA,Petroleum Product,None,0,302.0,Petroleum Products,None,None,None,2082.004459,pipeline_prod,1.293695,"MULTILINESTRING ((-97.38165 25.9667, -97.38265..."
4,5,TRANSMONTAIGNE PARTNERS,Diamondback Pipeline,EIA,Petroleum Product,None,0,303.0,Petroleum Products,None,None,None,9904.740005,pipeline_prod,6.154508,"MULTILINESTRING ((-97.52187 26.00233, -97.5283..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4307,4308,None,None,None,None,None,2,NaN,Petroleum Products,None,None,None,207.259275,pipeline_prod,0.128785,"MULTILINESTRING ((-93.19672 45.02409, -93.1989..."
4308,4309,None,None,None,None,None,2,NaN,Petroleum Products,None,None,None,242.718991,pipeline_prod,0.150818,"MULTILINESTRING ((-93.19672 45.02409, -93.1997..."
4309,4310,None,None,None,None,None,2,NaN,Petroleum Products,None,None,None,592.947623,pipeline_prod,0.368440,"MULTILINESTRING ((-93.19672 45.02409, -93.2032..."
4310,4311,None,None,None,None,None,2,NaN,Petroleum Products,None,None,None,836.752423,pipeline_prod,0.519933,"MULTILINESTRING ((-93.19672 45.02409, -93.1966..."


In [2]:
"""
ETL Transform Template.


This module provides a template for transforming raw data from multiple sources.
It includes standard cleaning, coercion, and normalization patterns used in the pipeline.
"""

import pandas as pd
import numpy as np
from typing import List, Optional, Dict
from prefect import task, get_run_logger
from ca_biositing.pipeline.utils.cleaning_functions import cleaning as cleaning_mod
from ca_biositing.pipeline.utils.cleaning_functions import coercion as coercion_mod
from ca_biositing.pipeline.utils.name_id_swap import normalize_dataframes

# --- CONFIGURATION ---
# List the names of the extract modules this transform depends on.
# The pipeline runner provides these in the `data_sources` dictionary.
EXTRACT_SOURCES: List[str] = ["petroleum_pipelines"]

@task
def transform(
    data_sources: Dict[str, pd.DataFrame],
    etl_run_id: int = None,
    lineage_group_id: int = None
) -> Optional[pd.DataFrame]:
    """
    Transforms raw data from multiple sources into a structured format.

    Args:
        data_sources: Dictionary where keys are source names and values are DataFrames.
        etl_run_id: ID of the current ETL run.
        lineage_group_id: ID of the lineage group.
    """
    try:
        logger = get_run_logger()
    except Exception:
        import logging
        logger = logging.getLogger(__name__)

    # CRITICAL: Lazy import models inside the task to avoid Docker import hangs
    from ca_biositing.datamodels.models import (
        Dataset,
        # Add other models needed for normalization here (e.g., Resource, Unit)
    )

    # 1. Input Validation
    for source_name in EXTRACT_SOURCES:
        if source_name not in data_sources:
            logger.error(f"Required data source '{source_name}' not found.")
            return None

    logger.info(f"Transforming data from sources: {EXTRACT_SOURCES}")

    # 2. Cleaning & Coercion
    processed_dfs = []
    for source_name in EXTRACT_SOURCES:
        df = data_sources[source_name].copy()

        if df.empty:
            continue

        # Standardize column names (snake_case) and basic string cleaning
        cleaned_df = cleaning_mod.standard_clean(df)

        # Add lineage tracking metadata
        cleaned_df['etl_run_id'] = etl_run_id
        cleaned_df['lineage_group_id'] = lineage_group_id

        # Coerce data types (Update these lists based on your schema)
        coerced_df = coercion_mod.coerce_columns(
            cleaned_df,
            int_cols=["artificial", "master_oid"],
            float_cols=["shape_length", "length"],
            datetime_cols=['created_at', 'updated_at']
        )
        processed_dfs.append(coerced_df)

    if not processed_dfs:
        return pd.DataFrame()

    # Combine sources if necessary, or handle them individually
    combined_df = pd.concat(processed_dfs, ignore_index=True)

    # 3. Normalization (Name-to-ID Swapping)
    # Format: 'dataframe_column': (SQLAlchemyModel, 'lookup_field_in_db')
    normalize_columns = {
        # 'dataset': (Dataset, 'name'),
        # Example: 'unit': (Unit, 'name'),
    }

    logger.info("Normalizing data (swapping names for IDs)...")
    normalized_df = normalize_dataframes(combined_df, normalize_columns)[0]

    # 4. Column Renaming
    # TODO: Update this dictionary to match your source-to-target mapping
    rename_columns = {
        'objectid': 'object_id',
        "opername": "operator_name",
        "pipename": "pipeline_name",
        "type": "pipeline_type",
        "geometry": "geom"
    }
    normalized_df = normalized_df.rename(columns=rename_columns)

    # 5. Final Mapping & Selection
    # TODO: Update this list to match the columns in your target database table
    try:
        # Ensure lineage columns exist even if not provided in input
        if etl_run_id:
            normalized_df['etl_run_id'] = etl_run_id
        if lineage_group_id:
            normalized_df['lineage_group_id'] = lineage_group_id

        final_df = normalized_df[[
            "object_id",
            "operator_name",
            "pipeline_name",
            "source", 
            "pipeline_type",
            "notes",
            "artificial",
            "master_oid",
            "commodity",
            "volume",
            "capacity",
            "vcr",
            "shape_length",
            "mode_type",
            "length",
            "geom"
        ]].copy()

        logger.info(f"Successfully transformed {len(final_df)} records.")
        return final_df

    except KeyError as e:
        logger.error(f"Missing required column during transform: {e}")
        return normalized_df

transformed_df = transform({"petroleum_pipelines": json_df})

14:30:05.632 | INFO    | prefect - Starting temporary server on http://127.0.0.1:8101
See https://docs.prefect.io/v3/concepts/server#how-to-guides for more information on running a dedicated Prefect server.

RuntimeError: Timed out while attempting to connect to ephemeral Prefect API server.

In [5]:
import pandas as pd
import numpy as np
from datetime import datetime, timezone
from prefect import task, get_run_logger
from sqlalchemy.dialects.postgresql import insert
from sqlalchemy.orm import Session
from ca_biositing.pipeline.utils.engine import get_engine

@task
def load_data(df: pd.DataFrame):
    """
    Upserts records into the database.

    Template Instructions:
    1. Replace 'YourModel' with the actual SQLModel class.
    2. Update 'index_elements' with the unique constraint column (e.g., 'record_id').
    3. Adjust 'update_dict' exclusions as needed.
    """
    try:
        logger = get_run_logger()
    except Exception:
        import logging
        logger = logging.getLogger(__name__)

    if df is None or df.empty:
        logger.info("No data to load.")
        return

    logger.info(f"Upserting {len(df)} records...")

    try:
        # CRITICAL: Lazy import models inside the task to avoid Docker import hangs
        # from ca_biositing.datamodels.models import YourModel
        from ca_biositing.datamodels.models.infrastructure.petroleum_pipelines import InfrastructurePetroleumPipelines as PetroPipelines # Placeholder

        now = datetime.now(timezone.utc)

        # Filter columns to match the table schema
        table_columns = {c.name for c in PetroPipelines.__table__.columns}
        records = df.replace({np.nan: None}).to_dict(orient='records')

        engine = get_engine()
        with engine.connect() as conn:
            with Session(bind=conn) as session:
                for i, record in enumerate(records):
                    # Optional: Add progress logging for larger datasets (from landiq.py)
                    if i > 0 and i % 500 == 0:
                        logger.info(f"Processed {i} records...")

                    # Clean record to only include valid table columns
                    clean_record = {k: v for k, v in record.items() if k in table_columns}

                    # Handle timestamps
                    clean_record['updated_at'] = now
                    if clean_record.get('created_at') is None:
                        clean_record['created_at'] = now

                    # Build Upsert Statement (PostgreSQL specific)
                    stmt = insert(PetroPipelines).values(clean_record)

                    # Define columns to update on conflict
                    # Exclude primary keys and creation timestamps
                    update_dict = {
                        c.name: stmt.excluded[c.name]
                        for c in PetroPipelines.__table__.columns
                        if c.name not in ['created_at', 'object_id']
                    }

                    upsert_stmt = stmt.on_conflict_do_update(
                        index_elements=['object_id'], # Replace with your unique constraint column
                        set_=update_dict
                    )

                    session.execute(upsert_stmt)

                session.commit()
        logger.info("Successfully upserted records.")
    except Exception as e:
        logger.error(f"Failed to load records: {e}")
        raise

load_data(transformed_df)


17:28:08.125 | INFO    | Task run 'load_data' - Upserting 4312 records...

17:28:12.471 | ERROR   | Task run 'load_data' - Failed to load records: (psycopg2.OperationalError) connection to server at "localhost" (::1), port 5432 failed: Connection refused (0x0000274D/10061)
        Is the server running on that host and accepting TCP/IP connections?
connection to server at "localhost" (127.0.0.1), port 5432 failed: Connection refused (0x0000274D/10061)
        Is the server running on that host and accepting TCP/IP connections?

(Background on this error at: https://sqlalche.me/e/20/e3q8)

17:28:12.479 | ERROR   | Task run 'load_data' - Task run failed with exception: OperationalError('(psycopg2.OperationalError) connection to server at "localhost" (::1), port 5432 failed: Connection refused (0x0000274D/10061)\n\tIs the server running on that host and accepting TCP/IP connections?\nconnection to server at "localhost" (127.0.0.1), port 5432 failed: Connection refused (0x0000274D/10061)\n\tIs the server running on that host and accepting TCP/IP connections?\n')
Traceback (most recent call last):
  File "C:\Users\Abigail\OneDrive\Documents\GitHub\ca-biositing\.pixi\envs\default\Lib\site-packages\sqlalchemy\engine\base.py", line 143, in __init__
    self._dbapi_connection = engine.raw_connection()
                             ~~~~~~~~~~~~~~~~~~~~~^^
  File "C:\Users\Abigail\OneDrive\Documents\GitHub\ca-biositing\.pixi\envs\default\Lib\site-packages\sqlalchemy\engine\base.py", line 3317, in raw_connection
    return self.pool.connect()
           ~~~~~~~~~~~~~~~~~^^
  File "C:\Users\Abigail\OneDrive\Documents\GitHub\ca-biositing\.pixi\envs\default\Lib\site-packages\sqlalchemy\pool\base.py", line 448, in connect
    return _ConnectionFairy._checkout(self)
           ~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^
  File "C:\Users\Abigail\OneDrive\Documents\GitHub\ca-biositing\.pixi\envs\default\Lib\site-packages\sqlalchemy\pool\base.py", line 1272, in _checkout
    fairy = _ConnectionRecord.checkout(pool)
  File "C:\Users\Abigail\OneDrive\Documents\GitHub\ca-biositing\.pixi\envs\default\Lib\site-packages\sqlalchemy\pool\base.py", line 712, in checkout
    rec = pool._do_get()
  File "C:\Users\Abigail\OneDrive\Documents\GitHub\ca-biositing\.pixi\envs\default\Lib\site-packages\sqlalchemy\pool\impl.py", line 177, in _do_get
    with util.safe_reraise():
         ~~~~~~~~~~~~~~~~~^^
  File "C:\Users\Abigail\OneDrive\Documents\GitHub\ca-biositing\.pixi\envs\default\Lib\site-packages\sqlalchemy\util\langhelpers.py", line 121, in __exit__
    raise exc_value.with_traceback(exc_tb)
  File "C:\Users\Abigail\OneDrive\Documents\GitHub\ca-biositing\.pixi\envs\default\Lib\site-packages\sqlalchemy\pool\impl.py", line 175, in _do_get
    return self._create_connection()
           ~~~~~~~~~~~~~~~~~~~~~~~^^
  File "C:\Users\Abigail\OneDrive\Documents\GitHub\ca-biositing\.pixi\envs\default\Lib\site-packages\sqlalchemy\pool\base.py", line 389, in _create_connection
    return _ConnectionRecord(self)
  File "C:\Users\Abigail\OneDrive\Documents\GitHub\ca-biositing\.pixi\envs\default\Lib\site-packages\sqlalchemy\pool\base.py", line 674, in __init__
    self.__connect()
    ~~~~~~~~~~~~~~^^
  File "C:\Users\Abigail\OneDrive\Documents\GitHub\ca-biositing\.pixi\envs\default\Lib\site-packages\sqlalchemy\pool\base.py", line 900, in __connect
    with util.safe_reraise():
         ~~~~~~~~~~~~~~~~~^^
  File "C:\Users\Abigail\OneDrive\Documents\GitHub\ca-biositing\.pixi\envs\default\Lib\site-packages\sqlalchemy\util\langhelpers.py", line 121, in __exit__
    raise exc_value.with_traceback(exc_tb)
  File "C:\Users\Abigail\OneDrive\Documents\GitHub\ca-biositing\.pixi\envs\default\Lib\site-packages\sqlalchemy\pool\base.py", line 896, in __connect
    self.dbapi_connection = connection = pool._invoke_creator(self)
                                         ~~~~~~~~~~~~~~~~~~~~^^^^^^
  File "C:\Users\Abigail\OneDrive\Documents\GitHub\ca-biositing\.pixi\envs\default\Lib\site-packages\sqlalchemy\engine\create.py", line 667, in connect
    return dialect.connect(*cargs_tup, **cparams)
           ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Abigail\OneDrive\Documents\GitHub\ca-biositing\.pixi\envs\default\Lib\site-packages\sqlalchemy\engine\default.py", line 630, in connect
    return self.loaded_dbapi.connect(*cargs, **cparams)  # type: ignore[no-any-return]  # NOQA: E501
           ~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Abigail\OneDrive\Documents\GitHub\ca-biositing\.pixi\envs\default\Lib\site-packages\psycopg2\__init__.py", line 135, 

C:\Users\Abigail\OneDrive\Documents\GitHub\ca-biositing\.pixi\envs\default\Lib\logging\__init__.py:1946: UserWarning: Logger 'prefect.task_runs' attempted to send logs to the API without a flow run id. The API log handler can only send logs within flow run contexts unless the flow run id is manually provided. Set PREFECT_LOGGING_TO_API_WHEN_MISSING_FLOW=ignore to suppress this warning.
  self.logger.log(level, msg, *args, **kwargs)


17:28:12.651 | ERROR   | Task run 'load_data' - Finished in state Failed('Task run encountered an exception OperationalError: (psycopg2.OperationalError) connection to server at "localhost" (::1), port 5432 failed: Connection refused (0x0000274D/10061)\n\tIs the server running on that host and accepting TCP/IP connections?\nconnection to server at "localhost" (127.0.0.1), port 5432 failed: Connection refused (0x0000274D/10061)\n\tIs the server running on that host and accepting TCP/IP connections?\n\n(Background on this error at: https://sqlalche.me/e/20/e3q8)')

OperationalError: (psycopg2.OperationalError) connection to server at "localhost" (::1), port 5432 failed: Connection refused (0x0000274D/10061)
	Is the server running on that host and accepting TCP/IP connections?
connection to server at "localhost" (127.0.0.1), port 5432 failed: Connection refused (0x0000274D/10061)
	Is the server running on that host and accepting TCP/IP connections?

(Background on this error at: https://sqlalche.me/e/20/e3q8)